# Register PacePilot agent preview
Read-only deterministic tools; no LLM, SQL connection, external actions, or secrets.
Upload `pacepilot-agent-code.zip` to the ROOT of `/Volumes/workspace/pacepilot/telemetry` before running.
The archive must not be in `incoming`, which is reserved for telemetry JSON.
This notebook registers a model. It does not create a serving endpoint.


In [ ]:
%pip install "mlflow[databricks]>=3,<4" "pydantic>=2.10,<3"


In [ ]:
dbutils.library.restartPython()


In [ ]:
import sys
import tempfile
import zipfile
from pathlib import Path

bundle = Path('/Volumes/workspace/pacepilot/telemetry/pacepilot-agent-code.zip')
if not bundle.exists():
    raise FileNotFoundError('Upload pacepilot-agent-code.zip to the telemetry volume root first')
code_root = Path(tempfile.mkdtemp(prefix='pacepilot-agent-'))
allowed = {'agent/__init__.py','agent/serving_model.py','shared/__init__.py','shared/schemas.py','shared/metrics.py'}
with zipfile.ZipFile(bundle) as archive:
    if set(archive.namelist()) != allowed:
        raise ValueError('Unexpected archive contents; use the generated project bundle')
    archive.extractall(code_root)
sys.path.insert(0, str(code_root))
from agent.serving_model import WearableAgentModel
print('Agent code loaded')


In [ ]:
import json
import pandas as pd
import mlflow
from mlflow.models import infer_signature

model_name = 'workspace.pacepilot.wearable_agent'
mlflow.set_registry_uri('databricks-uc')
user = spark.sql('SELECT current_user() AS username').first()['username']
mlflow.set_experiment(f'/Users/{user}/pacepilot-agent')

events = [
    {'schema_version':'1.0','user_id':'demo-athlete','device_id':'mock-polar-h10',
     'session_id':'serving-smoke-test','sequence':i,'timestamp':f'2026-10-03T12:00:0{i}Z',
     'heart_rate':75,'rr_intervals_ms':rr,'source':'mock','sensor_contact':True}
    for i,rr in enumerate([[800,810],[790,805]])
]
example = pd.DataFrame([{'prompt':'Summarize my session','telemetry_json':json.dumps(events)}])
model = WearableAgentModel()
expected = model.predict(None, example)
assert json.loads(expected[0]['response_json'])['metrics']['rr_count'] == 4
with mlflow.start_run():
    info = mlflow.pyfunc.log_model(
        name='agent', python_model=model,
        code_paths=[str(code_root / 'agent'), str(code_root / 'shared')],
        input_example=example, signature=infer_signature(example, expected),
        pip_requirements=[f'mlflow=={mlflow.__version__}', 'pydantic>=2.10,<3', f'pandas=={pd.__version__}'])
    loaded = mlflow.pyfunc.load_model(info.model_uri)
    prediction = loaded.predict(example)
    assert json.loads(prediction[0]['response_json'])['metrics']['rr_count'] == 4
    version = mlflow.register_model(info.model_uri, model_name)
print(f'REGISTERED MODEL: {model_name}')
print(f'REGISTERED VERSION: {version.version}')
print('Preview result:', expected[0]['response_json'])


After registration succeeds, create a custom Model Serving endpoint for the printed model/version.
Use CPU Small and scale to zero if those controls are available.
Request format: `{"dataframe_records":[{"prompt":"Summarize my session","telemetry_json":"[...]"}]}`.
The prompt is reserved for future LLM reasoning; this version runs the HRV tool deterministically.
